Feature Engineering

By: Maria Gonzalez

Date: 9.18.26-9.26.26

This notebook documents the feature engineering step for the Break Through Tech Healthcare 1E project.


Notes for running this notebook:

This notebook can be run independently from the notebooks/ folder from the Github. Make sure the repo has the standard folder structure and that the raw data file kidney_waitlist_analytic.csv.gz is located in ../data/. Then open this notebook and run it from top to bottom. You do not need to run any other notebooks or create any intermediate files first.

In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

Load Data

In [8]:
RAW_DATA_PATH = "../data/kidney_waitlist_analytic.csv.gz"

df_raw = pd.read_csv(RAW_DATA_PATH, low_memory=False)
print(df_raw.shape)
df_raw.head()

(494862, 38)


,ON_DIALYSIS,A2A2B_ELIGIBILITY,GENDER,ABO,BMI_TCR,FUNC_STAT_TCR,INIT_STAT,INIT_CPRA,END_CPRA,REM_CD,...,PSTATUS,PTIME,TRR_ID_CODE,DONOR_ID,LISTING_CTR_CODE,outcome,event_adverse,event_transplant,censored,days_to_event
0,Y,NaN,F,B,31.63,2080.0,4099,0.0,0.0,8.0,...,NaN,NaN,NaN,NaN,13609,died,1,0,0,287.0
1,Y,NaN,M,A,30.04,2070.0,4099,0.0,NaN,24.0,...,NaN,NaN,NaN,NaN,6975,removed_administrative,0,0,0,2322.0
2,N,NaN,F,O,32.85,2070.0,4099,0.0,0.0,8.0,...,NaN,NaN,NaN,NaN,19716,died,1,0,0,604.0
3,Y,NaN,M,A,20.00,2090.0,4099,0.0,0.0,13.0,...,NaN,NaN,NaN,NaN,8587,removed_too_sick,1,0,0,909.0
4,Y,NaN,M,AB,23.30,2070.0,4010,0.0,0.0,13.0,...,NaN,NaN,NaN,NaN,18352,removed_too_sick,1,0,0,231.0


Extract Relevant Columns for Feature Engineering

Relevant Columns:
- Same features used in encoding_scaling.ipynb (task #3) based on data_dictionary.md
- INIT_DATE is kept in this notebook so we can extract policy_era later
- event_adverse is kept as a check for the feature engineering, but is dropped before the final data export!

In [9]:
feature_cols = [
    'ON_DIALYSIS', 'A2A2B_ELIGIBILITY', 'GENDER', 'ABO', 'BMI_TCR',
    'FUNC_STAT_TCR', 'INIT_STAT', 'INIT_CPRA', 'INIT_AGE',
    'DIALYSIS_DATE', 'INIT_DATE', 'ETHCAT', 'REGION'
]

target_col = 'event_adverse'  # kept for validation checks only, not used as a feature

df_model = df_raw[feature_cols + [target_col]].copy()
df_model.head()

,ON_DIALYSIS,A2A2B_ELIGIBILITY,GENDER,ABO,BMI_TCR,FUNC_STAT_TCR,INIT_STAT,INIT_CPRA,INIT_AGE,DIALYSIS_DATE,INIT_DATE,ETHCAT,REGION,event_adverse
0,Y,NaN,F,B,31.63,2080.0,4099,0.0,53,2018-03-30,2020-03-25,2,8,1
1,Y,NaN,M,A,30.04,2070.0,4099,0.0,56,2017-08-16,2020-02-14,1,5,0
2,N,NaN,F,O,32.85,2070.0,4099,0.0,47,NaN,2020-05-27,1,11,1
3,Y,NaN,M,A,20.00,2090.0,4099,0.0,61,2019-01-05,2020-04-02,5,7,1
4,Y,NaN,M,AB,23.30,2070.0,4010,0.0,61,2019-01-10,2020-02-05,2,7,1


Cleaning steps (based on previous notebook).

- This code block mirrors the cleaning steps from Task #3, so that this can be run independently.

In [ ]:
# cap bmi at 80, per encoding_scaling.ipynb
df_model.loc[df_model['BMI_TCR'] > 80, 'BMI_TCR'] = df_model['BMI_TCR'].median()
df_model['BMI_TCR'].describe()

In [10]:
scale_mapping_dict = {}
percent_mapping_dict = {}

scale_mapping_dict[-1] = 'missing'
percent_mapping_dict[-1] = None

scale_mapping_dict[1] = 'adl_independent'
percent_mapping_dict[1] = None

scale_mapping_dict[996] = 'not_applicable'
percent_mapping_dict[996] = None

scale_mapping_dict[998] = 'unknown'
percent_mapping_dict[998] = None

# adult Karnofsky: 2010-2100
for x in range(2010, 2101, 10):
    scale_mapping_dict[x] = 'adult_karnofsky'
    percent_mapping_dict[x] = x - 2000

# pediatric Lansky
for x in range(4010, 4101, 10):
    scale_mapping_dict[x] = 'pediatric_lansky'
    percent_mapping_dict[x] = x - 4000

df_model['functional_scale'] = df_model['FUNC_STAT_TCR'].map(scale_mapping_dict)
df_model['functional_percent'] = df_model['FUNC_STAT_TCR'].map(percent_mapping_dict)

df_model['functional_scale'] = df_model['functional_scale'].fillna('other/unresolved')
df_model['functional_percent'] = df_model['functional_percent'].fillna(df_model['functional_percent'].median())

df_model = df_model.drop(columns=['FUNC_STAT_TCR'])
df_model[['functional_scale', 'functional_percent']].head()

,functional_scale,functional_percent
0,adult_karnofsky,80.0
1,adult_karnofsky,70.0
2,adult_karnofsky,70.0
3,adult_karnofsky,90.0
4,adult_karnofsky,70.0


In [11]:
#For dialysis duration: days on dialysis before listing
df_model['DIALYSIS_DATE_cleaned'] = pd.to_datetime(df_model['DIALYSIS_DATE'], errors='coerce')

df_model['dialysis_duration_days'] = (
    pd.to_datetime(df_model['INIT_DATE'], errors='coerce') - df_model['DIALYSIS_DATE_cleaned']
).dt.days

#If there's no dialysis date on record, we treat it as 0 days on dialysis before listing
df_model['dialysis_duration_days'] = df_model['dialysis_duration_days'].fillna(0)

#If there's negative durations (dialysis started after listing), we set them as 0, since we're answering
# "how many days had this patient been on dialysis before listing"
df_model.loc[df_model['dialysis_duration_days'] < 0, 'dialysis_duration_days'] = 0

df_model['dialysis_duration_days'].describe()

,dialysis_duration_days
count,494862.000000
mean,582.238202
std,918.104598
min,0.000000
25%,0.000000
50%,247.000000
75%,762.000000
max,15356.000000


policy_era feature:
- The dataset spans multiple OPTN allocation-policy eras (i.e. the 2014 Kidney Allocation System change).
- Bucketing each listing by era will let our models learn era-specific patterns instead of conflating them.
- Explicitly flagged as a distribution-shift risk in the project brief.

In [12]:
df_model['INIT_DATE'] = pd.to_datetime(df_model['INIT_DATE'], errors='coerce')

def get_policy_era(listing_date):
    if pd.isna(listing_date):
        return 'unknown'
    elif listing_date < pd.Timestamp('2015-01-01'):
        return 'pre_2015'
    elif listing_date < pd.Timestamp('2021-01-01'):
        return '2015_2020'
    else:
        return 'post_2021'

df_model['policy_era'] = df_model['INIT_DATE'].apply(get_policy_era)
df_model['policy_era'].value_counts()

,count
policy_era,
post_2021,262486
2015_2020,232376


Checking if there's a difference event_adverse rate by era:

In [13]:
df_model.groupby('policy_era')[target_col].mean()

,event_adverse
policy_era,
2015_2020,0.201363
post_2021,0.087437


cpra_missing feature:
- INIT_CPRA has substantial missing data per the data dictionary.
- Might be important to construct this feature.

In [14]:
df_model['cpra_missing'] = df_model['INIT_CPRA'].isna().astype(int)
df_model['cpra_missing'].value_counts()

,count
cpra_missing,
0,352121
1,142741


Check impoact of cpra_missing:

In [15]:
df_model.groupby('cpra_missing')[target_col].mean()

,event_adverse
cpra_missing,
0,0.174522
1,0.058077


bmi_category feature:
-  a clinical BMI bucket may help tree-based models capture risks for extreme values

In [16]:
df_model['bmi_category'] = pd.cut(
    df_model['BMI_TCR'],
    bins=[0, 18.5, 25, 30, 100],
    labels=['underweight', 'normal', 'overweight', 'obese']
)
df_model['bmi_category'].value_counts()

,count
bmi_category,
obese,200628
overweight,158488
normal,121008
underweight,12917


In [17]:
df_model.groupby('bmi_category', observed=True)[target_col].mean()

,event_adverse
bmi_category,
underweight,0.079585
normal,0.126620
overweight,0.144232
obese,0.150502


age_x_functional feature:
- Patient age and functional status percent interaction may produce more value than either factor individually.

In [18]:
df_model['age_x_functional'] = df_model['INIT_AGE'] * df_model['functional_percent']
df_model['age_x_functional'].describe()

,age_x_functional
count,494862.000000
mean,3988.209986
std,1323.070493
min,0.000000
25%,3100.000000
50%,4060.000000
75%,4950.000000
max,9000.000000


In [19]:
df_model[['age_x_functional', target_col]].corr()

,age_x_functional,event_adverse
age_x_functional,1.000000,0.097868
event_adverse,0.097868,1.000000


 Encode categorical variables and scale numeric variables:

In [21]:
categorical_cols = [
    'ON_DIALYSIS', 'GENDER', 'ABO', 'A2A2B_ELIGIBILITY',
    'INIT_STAT', 'ETHCAT', 'REGION', 'functional_scale',
    'policy_era', 'bmi_category'
]

numerical_cols = [
    'BMI_TCR', 'INIT_CPRA', 'INIT_AGE', 'functional_percent',
    'dialysis_duration_days', 'age_x_functional'
]

df_encoded = pd.get_dummies(df_model, columns=categorical_cols, drop_first=True)

scaler = StandardScaler()
df_encoded[numerical_cols] = scaler.fit_transform(df_encoded[numerical_cols])

df_encoded.head()

cols_to_drop = [c for c in ['INIT_DATE', 'DIALYSIS_DATE', 'DIALYSIS_DATE_cleaned'] if c in df_encoded.columns]
df_encoded = df_encoded.drop(columns=cols_to_drop)

df_encoded.shape

(494862, 48)

Export data set with feature engineering done.

In [22]:
import os

os.makedirs('../data/processed', exist_ok=True)
output_path = '../data/processed/engineered_features.csv'

df_encoded.to_csv(output_path, index=False)
print(f"Saved {df_encoded.shape[0]} rows and {df_encoded.shape[1]} columns to {output_path}")

Saved 494862 rows and 48 columns to ../data/processed/engineered_features.csv
